## Olink Cross-sectional data selection

### Aim 1: Healthy (CON) | At-risk (ARI)

Batch outliers = proteins > 2 SD from mean comparing last BRI batch with initial BRI batches (see Aim 3 for more details)

Valid value exclusions = list of proteins excluded because > 50% of their NPX values were below LOD for either At-risk or HC populations.

In [1]:
quiet_library <- function(...) { suppressPackageStartupMessages(library(...)) }

In [2]:
quiet_library(hise)
quiet_library(dplyr)

In [3]:
if(!dir.exists("output")) {
    dir.create("output")
}

In [4]:
cache_uuid_path <- function(uuid) {
    cache_path = paste0("cache/", uuid)
    if(!dir.exists(cache_path)) {
        cacheFiles(list(uuid))
    }
    cache_file = list.files(cache_path, full.names = TRUE)
    cache_file
}

## Read all RA Olink data

In [5]:
in_uuid <- "f6f5df5f-453e-4554-b28d-ba20f399ae69"

In [6]:
olink_file <- cache_uuid_path(in_uuid)
olink <- read.csv(olink_file)

[1] "Initiating file download for 2023-11-10_Olink_RAonly_noLabs.csv"
[1] "Download successful."


Failed to download files:



## Read batch outliers

In [7]:
outlier_uuid <- "be1822b9-84d9-4918-be25-2919e980955f"

In [8]:
outlier_file <- cache_uuid_path(outlier_uuid)
outliers <- read.csv(outlier_file, row.names = 1)

[1] "Initiating file download for batch-outliers-q04084_zcore2_filtered.csv"
[1] "Download successful."


Failed to download files:



In [9]:
nrow(outliers)

[1] 15

## Select data to retain for modeling

Read Sample IDs from HISE

In [10]:
samples_uuid <- "65472b32-ba00-4aab-9b3b-2309f2c57cf5"

In [11]:
samples_file <- cache_uuid_path(samples_uuid)
samples <- read.csv(samples_file)
nrow(samples)

[1] "Initiating file download for Aim1_Olink_Kits.csv"
[1] "Download successful."


Failed to download files:



[1] 82

Read Metadata from HISE

In [12]:
meta_uuid <- "32b8bb9c-4834-4e5c-9b3a-7939b3b7f057"

In [13]:
meta_file <- cache_uuid_path(meta_uuid)
meta <- read.csv(meta_file)

[1] "Initiating file download for 2023-11-22_ALTRA_Metadata_labs.csv"
[1] "Download successful."


Failed to download files:



In [14]:
keep_meta <- meta %>%
  filter(sample.sampleKitGuid %in% samples$sample.sampleKitGuid)

In [15]:
keep_olink <- olink %>%
  filter(sample.sampleKitGuid %in% samples$sample.sampleKitGuid)

Select panels to retain

In [16]:
keep_panels <- c("Cardiometabolic", "Inflammation", "Neurology", "Oncology")

In [17]:
keep_olink <- keep_olink %>%
  filter(Panel %in% keep_panels)

Remove outlier assays

In [18]:
keep_olink <- keep_olink %>%
  filter(!OlinkID %in% outliers$OlinkID)

In [19]:
nrow(keep_olink)

[1] 118910

Remove assays below LOD in at least 50% of either group

In [20]:
sample_status <- meta %>%
  select(sample.sampleKitGuid, Status_Xsec)

In [21]:
names(keep_olink)

[1] "SampleID"                   "Index"                     
 [3] "Assay_OID"                  "UniProt_OID"               
 [5] "OlinkID"                    "UniProt"                   
 [7] "Assay"                      "MissingFreq"               
 [9] "Panel"                      "Panel_Lot_Nr"              
[11] "PlateID_qry"                "QC_Warning_qry"            
[13] "sample.sampleKitGuid"       "Batch"                     
[15] "Bridged"                    "Assay_Warning"             
[17] "Manual_QC_Flag"             "Manual_QC_Note"            
[19] "NPX_Final"                  "LOD_Final"                 
[21] "subject.subjectGuid"        "sample.visitName"          
[23] "sample.visitDetails"        "sample.drawDate"           
[25] "sample.daysSinceFirstVisit" "subject.biologicalSex"     
[27] "subject.birthYear"          "subject.ethnicity"         
[29] "subject.race"               "cohort.cohortGuid"         
[31] "ageAtDraw"                  "timeOnStudy"               
[33] "timeOnStudyAsOf"

In [22]:
lod_summary <- keep_olink %>%
  left_join(sample_status) %>%
  group_by(OlinkID, Assay, Status_Xsec) %>%
  summarise(n_values = n(),
            n_below_lod = sum(NPX_Final < LOD_Final),
            frac_below_lod = n_below_lod / n_values,
            .groups = "keep")

Joining with `by = join_by(sample.sampleKitGuid)`


In [23]:
low_lod <- lod_summary %>%
  filter(frac_below_lod > 0.5)

In [24]:
keep_olink <- keep_olink %>%
  filter(!OlinkID %in% low_lod$OlinkID)

In [25]:
length(unique(keep_olink$OlinkID))

[1] 1327

Save filtered Olink data for downstream analysis

In [26]:
out_olink <- paste0(
    "output/",
    Sys.Date(),
    "_Olink_altraHCvARI.csv"
)

write.csv(
    keep_olink, 
    out_olink,
    row.names = FALSE
)

## Store results in HISE

In order to store the results in HISE, we'll need to cache these files to register them, and then we can upload the CSV files for later steps.

In [27]:
study_space_uuid <- "223de760-9624-45bd-aefe-ca24c75b1800"
title <- paste("ALTRA Olink Data for Aim 1: CON | ARI", Sys.Date())

In [28]:
search_id = ids::proquint(n_words = 3)
search_id

[1] "jihif-vurov-ditom"

In [29]:
in_list <- list(in_uuid, outlier_uuid, samples_uuid, meta_uuid)

In [30]:
out_list <- list(out_olink)

In [31]:
uploadFiles(
    files = out_list,
    studySpaceId = study_space_uuid,
    title = title,
    inputFileIds = in_list,
    destination = search_id,
    store = "project",
    doPrompt = FALSE
)

$files
$files[[1]]
[1] "output/2024-10-21_Olink_altraHCvARI.csv"


$traceId
[1] "20836676-8bd0-4ce8-a86f-bb0d925cc90a"

## Session Info

In [32]:
sessionInfo()

R version 4.3.2 (2023-10-31)
Platform: x86_64-conda-linux-gnu (64-bit)
Running under: Ubuntu 20.04.6 LTS

Matrix products: default
BLAS/LAPACK: /opt/conda/lib/libopenblasp-r0.3.25.so;  LAPACK version 3.11.0

locale:
 [1] LC_CTYPE=C.UTF-8       LC_NUMERIC=C           LC_TIME=C.UTF-8       
 [4] LC_COLLATE=C.UTF-8     LC_MONETARY=C.UTF-8    LC_MESSAGES=C.UTF-8   
 [7] LC_PAPER=C.UTF-8       LC_NAME=C              LC_ADDRESS=C          
[10] LC_TELEPHONE=C         LC_MEASUREMENT=C.UTF-8 LC_IDENTIFICATION=C   

time zone: Etc/UTC
tzcode source: system (glibc)

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
[1] dplyr_1.1.4 hise_2.16.0

loaded via a namespace (and not attached):
 [1] ids_1.0.1         crayon_1.5.2      vctrs_0.6.5       httr_1.4.7       
 [5] cli_3.6.3         rlang_1.1.4       stringi_1.8.3     generics_0.1.3   
 [9] assertthat_0.2.1  jsonlite_1.8.8    glue_1.7.0        RCurl_1.98-1.16  
[13] htmlt